-built multi agent travel agent system

-we will use 2 to 3 speciallized agents, like a destination selection agent , searches the location , hotels and books.

-another agent provides, weather and city information.

-may be a third agent uses all this info to plan travel itinery.

-build a sequential agent.

-reuse existing tools , and create new one required.

-club the each agent nodes using lang graph.

-get final output.

In [ ]:
# before built with lang graph create langchain chat pipeline
# from langchain_huggingface import HuggingFacePipeline,ChatHuggingFace
!pip install langchain-groq
import os
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY""
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0)

In [ ]:
#define the tools
import requests
from langchain_core.tools import tool

In [ ]:
# 1. weather tool - now using a real-time weather API (Open-Meteo, free, no API key needed)
import requests

@tool
def weather(city: str):
  """Returns the current real-time weather for a given city."""
  try:
    # step 1: turn the city name into latitude/longitude
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_res = requests.get(geo_url, params={"name": city, "count": 1}).json()
    if not geo_res.get("results"):
      return "weather data not found"

    lat = geo_res["results"][0]["latitude"]
    lon = geo_res["results"][0]["longitude"]

    # step 2: fetch the current weather for that location
    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_res = requests.get(weather_url, params={
        "latitude": lat,
        "longitude": lon,
        "current_weather": True
    }).json()

    current = weather_res.get("current_weather", {})
    temp = current.get("temperature")
    wind = current.get("windspeed")
    return f"{temp}C, wind {wind} km/h"
  except Exception as e:
    return f"weather data not found: {e}"


In [ ]:
# City info tool - now pulling a real summary from Wikipedia instead of a hardcoded dict
@tool
def Cityinfo(city: str):
  """Returns real city information for a given city, pulled from Wikipedia."""
  try:
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + city.strip().replace(" ", "_")
    headers = {"User-Agent": "MyTravelAgent/1.0 (myemail@example.com)"}
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
      return "city info data not found"

    data = res.json()
    return data.get("extract", "city info data not found")
  except Exception as e:
    return f"city info data not found: {e}"


In [ ]:
# Currency exchange tool - now using a real-time free API (frankfurter.app, backed by the European Central Bank)
@tool
def currency(amount: float, from_currency: str, to_currency: str):
  """Convert the amount from one currency to another using live exchange rates."""
  try:
    url = "https://api.frankfurter.app/latest"
    res = requests.get(url, params={"from": from_currency, "to": to_currency})
    if res.status_code != 200:
      return "Conversion not available"

    data = res.json()
    rate = data.get("rates", {}).get(to_currency)
    if rate is None:
      return "Conversion not available"

    return amount * rate
  except Exception as e:
    return f"Conversion not available: {e}"


In [ ]:
#dummy hotel list and then create hotel
MOCK_HOTELS={
    "udaipur":[
        {"name":"Lake Palace View","price_per_night":120},
        {"name":"City Palace Inn","price_per_night":70},
        {"name":"Budget Haveli Stay","price_per_night":35},

    ],
    "goa":[
        {"name":"Beachside Resort","price_per_night":150},
        {"name":"Palm Grove Hotel","price_per_night":60},
        {"name":"Backpacker Inn","price_per_night":25},
    ],
    "jaipur":[
        {"name":"Pink City Palace Hotel","price_per_night":100},
        {"name":"Heritage Haveli","price_per_night":55},
        {"name":"Traveler's Nest","price_per_night":30},
    ],
}

#define the hotel search tool
@tool
def hotel_search(city:str):
  "Returns a short list of mock hoteks for a given city,price per night in INR"
  hotels = MOCK_HOTELS.get(city.strip().lower())
  if not hotels:

    return f"No hotels info available for {city}"
  return hotels


In [ ]:
# calculator tool for budget computing
# use numexpr for computing
import numexpr
@tool
def calculator(expression:str):
  "Evaluate a basic arithmatic expression , eg. 3*129"
  try:
    return numexpr.evaluate(expression)
  except Exception as e:
    return f"error evaluating expression {expression} , {e}"

In [ ]:

# now build the agents 2 tool calling agents using creat_react_agent
from langgraph.prebuilt import create_react_agent

cityinfo_agent=create_react_agent(llm,tools=[weather,Cityinfo],
                                  prompt="""you are a cityinfo and weather agent report a short city summary
                                  and the current weather. Nothing else""")
booking_agent=create_react_agent(llm,tools=[hotel_search,currency,calculator],
                                 prompt="""you are a Hotel and Budget analyst. Pick 1 hotel(price in INR)
                                 use the calculator tool for nights*price_per_night, then use currency tool to
                                 convert the total. Never do this math yourself""")

/tmp/ipykernel_1280/3053637272.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  cityinfo_agent=create_react_agent(llm,tools=[weather,Cityinfo],
/tmp/ipykernel_1280/3053637272.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  booking_agent=create_react_agent(llm,tools=[hotel_search,currency,calculator],


In [ ]:
! pip install -q langchain-core langchain-groq langgraph requests


In [ ]:
#to create graph first create shared dictionary
#lang graph state

from typing import TypedDict
class TripState(TypedDict):
  city:str
  nights:int
  target_currency:str
  cityinfo_agent_summary:str
  booking_agent_summary:str
  final_plan:str

In [ ]:
from langchain_core.messages import AIMessage,HumanMessage
def cityinfo_node(state:TripState):
  result=cityinfo_agent.invoke({
      "messages":[
          HumanMessage(content=f"City:{state['city']}")]

  })
  return state | {"cityinfo_agent_summary":result["messages"][-1].content}
#define second node
def booking_node(state:TripState):
  prompt=(
    f"city:{state['city']},nights:{state['nights']}"
    f"convert the toal to:{state['target_currency']}")

  result=booking_agent.invoke({
      "messages":[
          HumanMessage(content=prompt)
      ]
  })
  return{"booking_agent_summary":result["messages"][-1].content}



In [ ]:
#final iteniary node
def iteniary_node(state:TripState):
  prompt=(
    f"Using the city/weather info:\n{state['cityinfo_agent_summary']}\n\n"
    f"and this hotel/budget info:\n{state['booking_agent_summary']}\n\n"
    f"write a day by day {state['nights']}-night itinerary for {state['city']}\n\n"
    "include the hotel pick and total cost in the final plan"
    f"Your itinerary must have exactly{state['nights']}day-rows,no more,no less"
  )
  result=llm.invoke([
          HumanMessage(content=prompt)
      ]
  )
  return{"final_plan":result.content}

# build the lang graph architecture

In [ ]:
from langgraph.graph import StateGraph ,START,END
graph=StateGraph(TripState)
#add nodes
graph.add_node("Cityinfo",cityinfo_node)
graph.add_node("booking",booking_node)
graph.add_node("itinerary",iteniary_node)
# now start connecting
graph.add_edge(START,"Cityinfo")
graph.add_edge("Cityinfo","booking")
graph.add_edge("booking","itinerary")
graph.add_edge("itinerary",END)
Travel_multi_agent=graph.compile()

In [ ]:
result=Travel_multi_agent.invoke({"city":"udaipur",
                                  "nights":3,
                                  "target_currency":"USD"})
print(result["final_plan"])

**3‑Night Itinerary – Udaipur**

| Day | Morning | Afternoon | Evening |
|-----|---------|-----------|---------|
| **Day 1** | Arrive in Udaipur and check‑in to **City Palace Inn**. Refresh and enjoy a leisurely walk around the nearby lakefront promenade. | Explore the historic **City Palace** complex and its museums (self‑guided). | Dine at a lakeside café and watch the sunset over Lake Pichola. |
| **Day 2** | Visit the **Jagdish Temple** and stroll through the bustling **Bada Bazaar** for local crafts. | Take a boat ride on Lake Pichola to see the **Lake Palace** and **Jag Mandir** from the water. | Attend a cultural dance performance at the **Bagore‑Ki‑Haveli**. |
| **Day 3** | Head to the **Saheliyon‑Ki‑Bari** gardens for a peaceful morning walk. | Explore the **Monsoon Palace** for panoramic views of the city and surrounding hills. | Enjoy a farewell dinner at a rooftop restaurant overlooking the illuminated cityscape. |

**Accommodation:** City Palace Inn  
**Total Cost:** $2.21

# Idea:User provides city,as input
- Agent checks weather,and curretn local news/events to flag anything relevant to travelers(rain,storms,festivals,bundh)
- weather agent-live forecast gathering
- news agent- fetches todays headlines for the given city/country,use google news or any news api
- advisor agent- LLM combines both into final advisory to go out today or not

# Movie night picker
- Mood agent: LLM maps your mood to a genere
- Movie Agent: uses genere to a realtime movie list(TMDB API) to suggest recent movies in the genere
- Availability agent: suggest where that movie is streaming,like netfliz,prime etc by region